# Pre-Processing

Runs the FET (Filter → Extract → Transform) pipeline on a raw acquisition directory
and saves the result as `.npy` checkpoint files.  The **Load from Checkpoint** section
at the bottom lets you reload saved files without re-running the pipeline.

In [ ]:
%load_ext autoreload
%autoreload 2

from naludaq.board import Board
from naluacq import Acquisition

import numpy as np
from pathlib import Path
import glob

from toolbox import *
from toolbox.plotting.interactive_plot import *
from toolbox.plotting.save_plots import *
from toolbox.data.utils import *
from toolbox.baseline import *
from toolbox.filters.pipelines import FETPipeline
from toolbox.filters.data_forming_filters import *
from toolbox.filters.array_builder import *
from toolbox.filters.templates import *


## Notes
Datasets collected at Berkeley EOS.

- TRBHM: [2025-10-22 10-03-47.951971](https://drive.google.com/drive/folders/1cN6eJw0Rpp8GpyTnQ9kpw3sB65hqh14h?usp=drive_link) (1408 events)
- ASOC: [2025-10-22 11-32-56.502022](https://drive.google.com/drive/folders/16gO0CscjxJqfUVxIQYBA4Vx00oQiK_yD?usp=drive_link) (10404 events)

Link to Whisper Data: https://drive.google.com/drive/folders/1QALq6wFnaYW2QnPZEMOzGRIXo8y3_oj0?usp=drive_link

## Configuration

| Variable | Description |
|---|---|
| `BOARD` | `"trbhm"` or `"asoc"` |
| `ACQ_DIR` | Root folder containing the acquisition sub-directories |
| `DATA_INDEX` | Which acquisition to process (see discovery cell output) |
| `EVENT_INDICES` | `None` to use all events, or a list/array of indices to restrict which events are loaded |
| `MIN_SAMPLES` | `None` to auto-detect from the dataset (samples the first 1000 events), or an explicit integer |
| `CHECKPOINT_FILE` | Base name for saved `.npy` files |
| `CHECKPOINT_FOLDER` | Where checkpoint files are written |
| `REPLACE_CHECKPOINT` | Overwrite existing files when `True` |

In [ ]:
BOARD = "trbhm"  # "trbhm" or "asoc"

ACQ_DIR = Path(r"D:\nalu\nalu-projects\data") / "WHISPER" / f"BerkeleyEOS_{BOARD.upper()}_2025Oct22"

DATA_INDEX = 1       # Dataset index to be pre-processed

EVENT_INDICES = np.arange(0, 1000)  # None = all events; e.g. list(range(1000)) for the first 1000

MIN_SAMPLES = None   # None = auto-detect; set an integer to override

CHECKPOINT_FILE   = f"{BOARD}-preprocessed"
CHECKPOINT_FOLDER = Path().cwd() / "data"
REPLACE_CHECKPOINT = True


CHECKPOINT_FOLDER.mkdir(exist_ok=True)

# Channel layout:
#   TRBHM : ch 0-6 signal, ch 7 trigger
#   ASOC  : ch 0-3 from chip-0, ch 4-7 from chip-1 (trigger on ch 7)
CHANNELS = list(range(8))

board = Board("trbhm") if BOARD == "trbhm" else Board("aodsoc_asoc")

## Board Summary

In [ ]:
print("Board Summary")
print("-------------")
print(f"Model:         {board.params['model']}")
print(f"Channels:      {board.params['channels']}")
print(f"Windows:       {board.params['windows']}")
print(f"Samples:       {board.params['samples']}")
print(f"Sampling Rate: {board.params['samplingrate']} GHz")


## Discover Acquisitions

Lists every acquisition directory under `ACQ_DIR`.
Use the printed index to set `DATA_INDEX` in the configuration cell above.

In [ ]:
dirs = sorted(d for d in glob.glob(str(ACQ_DIR) + "/*") if Path(d).is_dir())

acquisitions = []
print(f"{'Index':>5}  {'Events':>8}  Directory")
print("-" * 60)
for idx, d in enumerate(dirs):
    try:
        acq = Acquisition(d)
        acquisitions.append({"dir": d, "acq": acq})
        print(f"{idx:>5}  {len(acq):>8}  {Path(d).name}")
    except Exception as e:
        print(f"{idx:>5}  {'ERROR':>8}  {Path(d).name}  ({e})")


## Select Dataset

In [ ]:
entry   = acquisitions[DATA_INDEX]
DATASET = entry["acq"]

print(f"Selected : {Path(entry['dir']).name}")
print(f"Events   : {len(DATASET)}")


In [ ]:
if MIN_SAMPLES is None:
    probe_n = min(1000, len(DATASET))
    max_samples = 0
    for i in range(probe_n):
        try:
            samples = len(DATASET[i]["data"][CHANNELS[0]])
            if samples > max_samples:
                max_samples = samples
        except Exception:
            pass
    MIN_SAMPLES = max_samples
    print(f"Auto-detected MIN_SAMPLES = {MIN_SAMPLES}  (from first {probe_n} events)")
else:
    print(f"MIN_SAMPLES = {MIN_SAMPLES}  (manually set)")

## Lazy Conversion

The raw acquisition stores event fields as Python lists. The `convert_to_lazy_acq` function converts event fields to numpy arrays when accessed.

In [ ]:
def convert_to_lazy_acq(acq, indices=None):
    """Lazy wrapper: converts event fields to numpy arrays on access."""
    evt_keys = ["data", "window_labels", "time", "timing", "ecc_errors"]

    class LazyAcq:
        def __len__(self):
            return len(indices) if indices is not None else len(acq)

        def __getitem__(self, i):
            raw_i = indices[i] if indices is not None else i
            evt = acq[raw_i]
            return {k: np.asarray(evt[k]) for k in evt_keys}

    return LazyAcq()

DATASET_LAZY = convert_to_lazy_acq(DATASET, indices=EVENT_INDICES)
print(f"Lazy wrapper ready, {len(DATASET_LAZY)} events.")

## Load Pedestals

In [ ]:
peds = np.asarray(DATASET.pedestals_calibration["data"], dtype=np.float32)
print(f"Pedestals shape: {peds.shape}")


## Run FET Pipeline

Stage 0 filters out events that don't meet `MIN_SAMPLES`.  
`ArrayBuilder` writes directly into pre-allocated `(N, C, S)` arrays so RAM stays
bounded regardless of dataset size.

In [ ]:
pipeline = FETPipeline(channels=CHANNELS, pedestals=peds, check_timing=False)
pipeline.add_filter(0, MinDataLengthFilter(channels=CHANNELS, n_samples=MIN_SAMPLES))

processed = pipeline.run(DATASET_LAZY)

print(f"\nOutput keys : {list(processed.keys())}")
print(f"Data shape  : {processed['data'].shape}")


## Preview

In [ ]:
interactive_plot(processed["data"])


## Save Checkpoint

Each output key is saved as a separate `.npy` file.  Pedestals are included so the
checkpoint is self-contained.

In [ ]:
base = CHECKPOINT_FOLDER / CHECKPOINT_FILE

for key, arr in processed.items():
    out = base.parent / f"{base.name}-{key}.npy"
    if REPLACE_CHECKPOINT or not out.exists():
        np.save(out, arr)
        print(f"Saved  {out.name}  {np.asarray(arr).shape}")
    else:
        print(f"Skip   {out.name}  (exists, REPLACE_CHECKPOINT=False)")

peds_out = base.parent / f"{base.name}-peds.npy"
if REPLACE_CHECKPOINT or not peds_out.exists():
    np.save(peds_out, peds)
    print(f"Saved  {peds_out.name}  {peds.shape}")


## Load Checkpoint Data

In [ ]:
LOAD_BOARD = "trbhm"
LOAD_FILE  = f"{LOAD_BOARD}-preprocessed"
LOAD_DIR   = Path().cwd() / "data"

checkpoint = {}
for f in sorted(LOAD_DIR.glob(f"{LOAD_FILE}-*.npy")):
    key = f.stem.replace(f"{LOAD_FILE}-", "")
    try:
        checkpoint[key] = np.load(f, mmap_mode="r", allow_pickle=True)
        print(f"Loaded  {key:30s}  {checkpoint[key].shape}")
    except Exception:
        try:
            checkpoint[key] = np.load(f, allow_pickle=True)
            print(f"Loaded  {key:30s}")
        except Exception as e:
            print(f"Failed  {key}: {e}")

print(f"\n{len(checkpoint)} arrays loaded.")